In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import warnings
import logging
import pickle 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

DATA_PATH = "Data_new.csv"
MODEL_SAVE_DIR = "saved_models"
PLOT_SAVE_DIR = "saved_plots"
BEST_PARAMS_FILE = "best_params.json"
INPUT_CSV_PATH = "A-D-Aold.csv"
OUTPUT_CSV_PATH = "predicted_results_old.csv"
FOLD_RESULTS_DIR = "fold_results"
CROSS_VALIDATION_DIR = "cross_validation_results"
RESULTS_DATA_FILE = "results_data.pkl" 

for dir_path in [MODEL_SAVE_DIR, PLOT_SAVE_DIR, FOLD_RESULTS_DIR, CROSS_VALIDATION_DIR]:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        logging.info(f"Directory created: {dir_path}")

PARAM_BOUNDS = {
    'n_estimators': (50, 1000),       
    'learning_rate': (0.01, 0.3),     
    'max_depth': (3, 10),             
    'min_samples_split': (2, 20),      
    'min_samples_leaf': (1, 20),      
    'max_features': (0.3, 1.0),        
    'subsample': (0.5, 1.0)            
}

DEFAULT_BEST_PARAMS = {
    'n_estimators': 104,
    'learning_rate': 0.049664706208491917,             
    'max_depth': 3,                    
    'min_samples_split': 11,
    'min_samples_leaf': 2,
    'max_features': 0.6919328607479193,
    'subsample': 0.8,                  
    'random_state': 42
}

warnings.filterwarnings("ignore", category=UserWarning, message="Glyph.*")

plt.rcParams['font.family'] = ['Arial', 'SimHei', 'WenQuanYi Micro Hei', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family': ['Arial', 'SimHei'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 300,
    'figure.figsize': (3.3, 3),
    'lines.linewidth': 1,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5
})

NATURE_COLORS = [
    '#403990',
    '#80A6E2',
    '#FBDD85',
    '#F46F43',
    '#CF3D3E'
]

COLORS = {
    'primary': NATURE_COLORS[0],
    'secondary': NATURE_COLORS[3],
    'tertiary': NATURE_COLORS[2],
    'neutral': '#636363',
    'light': '#F7F7F7',
    'dark': '#252525',
    'ci_95': '#A6CEE3',
    'ci_90': '#B2DF8A',
    'cmap': LinearSegmentedColormap.from_list("nature_cmap", NATURE_COLORS)
}

def load_data():
    try:
        data_df = pd.read_csv(DATA_PATH, encoding='gbk')
        logging.info(f"Dataset loaded successfully, with {len(data_df)} records and {len(data_df.columns)} features")
        
        target_column = 'ET1'
        drop_columns = ['No.', 'Acceptor', target_column]
        feature_columns = [col for col in data_df.columns if col not in drop_columns]
        
        print("\n===== Feature and Target Column Information =====")
        print(f"Target column: {target_column}")
        print(f"Number of feature columns: {len(feature_columns)}")
        print(f"List of feature columns: {feature_columns}")
        print("===============================================\n")
        
        X = data_df[feature_columns]
        y = data_df[target_column]
        
        return X, y, data_df, feature_columns
    except FileNotFoundError:
        logging.error("Error: Dataset file not found, please check the file path.")
        raise

def save_results_data(results_data, file_path=RESULTS_DATA_FILE):
    try:
        with open(file_path, 'wb') as f:
            serializable_data = {k: v for k, v in results_data.items() 
                                if k not in ['final_model', 'scaler']}
            pickle.dump(serializable_data, f)
        logging.info(f"Result data saved to: {file_path}")
    except Exception as e:
        logging.error(f"Failed to save result data: {e}")

def load_results_data(file_path=RESULTS_DATA_FILE):
    try:
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        logging.error(f"Result data file not found: {file_path}, please run Module 2 first")
        raise
    except Exception as e:
        logging.error(f"Failed to load result data: {e}")
        raise

try:
    X, y, data_df, feature_names = load_data()
except Exception as e:
    logging.warning(f"Data preview failed: {e}, will retry in subsequent modules")

2025-10-11 16:09:45,996 - INFO - 数据集加载成功，共169条记录，6个特征



===== 特征列和目标列信息 =====
目标列: ET1
特征列数量: 3
特征列列表: ['new-AL', 'new-AR', 'new-D']



In [2]:
'''
import random
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

X, y, data_df, feature_names = load_data()

def init_ga():
    if not hasattr(creator, 'FitnessMax'):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    
    if not hasattr(creator, 'Individual'):
        creator.create("Individual", list, fitness=creator.FitnessMax)
    
    toolbox = base.Toolbox()
    
    toolbox.register("attr_n_estimators", random.randint, *PARAM_BOUNDS['n_estimators'])
    toolbox.register("attr_learning_rate", random.uniform, *PARAM_BOUNDS['learning_rate'])
    toolbox.register("attr_max_depth", random.randint, *PARAM_BOUNDS['max_depth'])
    toolbox.register("attr_min_samples_split", random.randint, *PARAM_BOUNDS['min_samples_split'])
    toolbox.register("attr_min_samples_leaf", random.randint, *PARAM_BOUNDS['min_samples_leaf'])
    toolbox.register("attr_max_features", random.uniform, *PARAM_BOUNDS['max_features'])
    toolbox.register("attr_subsample", random.uniform, *PARAM_BOUNDS['subsample'])
    
    toolbox.register("individual", tools.initCycle, creator.Individual,
                    (toolbox.attr_n_estimators, toolbox.attr_learning_rate,
                     toolbox.attr_max_depth, toolbox.attr_min_samples_split,
                     toolbox.attr_min_samples_leaf, toolbox.attr_max_features,
                     toolbox.attr_subsample), n=1)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    return toolbox

def custom_mutate(individual):
    mutation_probability = 0.2
    if random.random() < mutation_probability:
        individual[0] = random.randint(*PARAM_BOUNDS['n_estimators'])
    if random.random() < mutation_probability:
        individual[1] = random.uniform(*PARAM_BOUNDS['learning_rate'])
    if random.random() < mutation_probability:
        individual[2] = random.randint(*PARAM_BOUNDS['max_depth'])
    if random.random() < mutation_probability:
        individual[3] = random.randint(*PARAM_BOUNDS['min_samples_split'])
    if random.random() < mutation_probability:
        individual[4] = random.randint(*PARAM_BOUNDS['min_samples_leaf'])
    if random.random() < mutation_probability:
        individual[5] = random.uniform(*PARAM_BOUNDS['max_features'])
    if random.random() < mutation_probability:
        individual[6] = random.uniform(*PARAM_BOUNDS['subsample'])
    return individual,

def evaluate(params, X, y):
    param_dict = {
        'n_estimators': params[0],
        'learning_rate': params[1],
        'max_depth': params[2],
        'min_samples_split': params[3],
        'min_samples_leaf': params[4],
        'max_features': params[5],
        'subsample': params[6],
        'random_state': 42
    }
    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        scaler_fold = StandardScaler().fit(X_train_fold)
        X_train_fold_scaled = pd.DataFrame(scaler_fold.transform(X_train_fold), columns=X.columns)
        X_val_fold_scaled = pd.DataFrame(scaler_fold.transform(X_val_fold), columns=X.columns)

        model = GradientBoostingRegressor(**param_dict)
        model.fit(X_train_fold_scaled.values, y_train_fold)
        fold_scores.append(r2_score(y_val_fold, model.predict(X_val_fold_scaled.values)))

    return (np.mean(fold_scores),)

def optimize_hyperparameters(X, y, pop_size=50, generations=30, cxpb=0.7, mutpb=0.2):
    toolbox = init_ga()
    
    toolbox.register("evaluate", evaluate, X=X, y=y)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", custom_mutate)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    population = toolbox.population(n=pop_size)
    
    best_individuals = []
    
    logging.info("Starting hyperparameter optimization...")
    for gen in range(generations):
        offspring = algorithms.varAnd(population, toolbox, cxpb=cxpb, mutpb=mutpb)
        
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        
        population = toolbox.select(offspring, k=len(population))
        
        top_ind = tools.selBest(population, 1)[0]
        best_individuals.append((gen, top_ind, top_ind.fitness.values[0]))
        
        if gen % 5 == 0:
            logging.info(f"Generation {gen} - Best R²: {top_ind.fitness.values[0]:.4f}")
    
    best_gen, best_ind, best_score = max(best_individuals, key=lambda x: x[2])
    logging.info(f"\nOptimization complete! Best R² value: {best_score:.4f} (at generation {best_gen})")
    
    best_params = {
        'n_estimators': best_ind[0],
        'learning_rate': best_ind[1],
        'max_depth': best_ind[2],
        'min_samples_split': best_ind[3],
        'min_samples_leaf': best_ind[4],
        'max_features': best_ind[5],
        'subsample': best_ind[6],
        'random_state': 42
    }
    
    return best_params, best_score

best_params, best_score = optimize_hyperparameters(X, y)

with open(BEST_PARAMS_FILE, 'w') as f:
    json.dump(best_params, f, indent=4)

logging.info(f"Best hyperparameters saved to {BEST_PARAMS_FILE}:")
for param, value in best_params.items():
    logging.info(f"  {param}: {value}")
'''

'\n# 模块1：模型超参数筛选\n# 使用遗传算法优化GBDT模型的超参数\n\nimport random\nimport numpy as np\nimport pandas as pd\nfrom deap import base, creator, tools, algorithms\nfrom sklearn.model_selection import KFold\nfrom sklearn.metrics import r2_score\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.ensemble import GradientBoostingRegressor  # 导入GBDT模型\n\n# 加载数据\nX, y, data_df, feature_names = load_data()\n\n# 遗传算法初始化\ndef init_ga():\n    """初始化遗传算法相关组件"""\n    if not hasattr(creator, \'FitnessMax\'):\n        creator.create("FitnessMax", base.Fitness, weights=(1.0,))\n    \n    if not hasattr(creator, \'Individual\'):\n        creator.create("Individual", list, fitness=creator.FitnessMax)\n    \n    toolbox = base.Toolbox()\n    \n    # 注册参数生成器 - 调整为GBDT的参数\n    toolbox.register("attr_n_estimators", random.randint, *PARAM_BOUNDS[\'n_estimators\'])\n    toolbox.register("attr_learning_rate", random.uniform, *PARAM_BOUNDS[\'learning_rate\'])  # GBDT特有\n    toolbox.register("attr_max_depth", ran

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from joblib import dump
import json
import os
import scipy.stats as stats

X, y, data_df, feature_names = load_data()

try:
    with open(BEST_PARAMS_FILE, 'r') as f:
        best_params = json.load(f)
    logging.info(f"Loaded best parameters from {BEST_PARAMS_FILE}")
except FileNotFoundError:
    logging.warning(f"{BEST_PARAMS_FILE} not found, using default parameters")
    best_params = {
        'boosting_type': 'gbdt',
        'objective': 'regression',
        'metric': 'rmse',
        'n_estimators': 100,
        'learning_rate': 0.1,
        'max_depth': -1,
        'random_state': 9
    }

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=9)
logging.info(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")

scaler = StandardScaler().fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(X_train_scaled.values, y_train)

model_path = os.path.join(MODEL_SAVE_DIR, "lightgbm_model.joblib")
scaler_path = os.path.join(MODEL_SAVE_DIR, "scaler.joblib")
dump(final_model, model_path)
dump(scaler, scaler_path)
logging.info(f"Model saved to: {model_path}")
logging.info(f"Scaler saved to: {scaler_path}")

y_pred_test = final_model.predict(X_test_scaled.values)
test_metrics = {
    'MAE': mean_absolute_error(y_test, y_pred_test),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'R²': r2_score(y_test, y_pred_test),
    'r': np.corrcoef(y_test, y_pred_test)[0][1]
}

y_pred_train = final_model.predict(X_train_scaled.values)
train_metrics = {
    'MAE': mean_absolute_error(y_train, y_pred_train),
    'RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
    'R²': r2_score(y_train, y_pred_train),
    'r': np.corrcoef(y_train, y_pred_train)[0][1]
}

logging.info(f"\nTest set metrics: R²={test_metrics['R²']:.4f}, MAE={test_metrics['MAE']:.4f}, RMSE={test_metrics['RMSE']:.4f}, r={test_metrics['r']:.4f}")
logging.info(f"Training set metrics: R²={train_metrics['R²']:.4f}, MAE={train_metrics['MAE']:.4f}, RMSE={train_metrics['RMSE']:.4f}, r={train_metrics['r']:.4f}")

def calculate_confidence_interval(model, X, y_true, y_pred, alpha=0.05):
    residuals = y_true - y_pred
    residual_std = np.std(residuals)
    n = len(y_true)
    t_value = stats.t.ppf(1 - alpha/2, n - 2)
    prediction_interval = t_value * residual_std * np.sqrt(1 + 1/n)
    lower_bound = y_pred - prediction_interval
    upper_bound = y_pred + prediction_interval
    return lower_bound, upper_bound, prediction_interval

lower_bound_test, upper_bound_test, _ = calculate_confidence_interval(
    final_model, X_test_scaled.values, y_test.values, y_pred_test
)
lower_bound_train, upper_bound_train, _ = calculate_confidence_interval(
    final_model, X_train_scaled.values, y_train.values, y_pred_train
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_metrics = []
all_y_true = []
all_y_pred = []
all_lower_bounds = []
all_upper_bounds = []
all_indices = []

logging.info("\nStarting 10-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    logging.info(f"\n{'='*20} Fold {fold} {'='*20}")
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    scaler_fold = StandardScaler().fit(X_train_fold)
    X_train_fold_scaled = pd.DataFrame(scaler_fold.transform(X_train_fold), columns=feature_names)
    X_val_fold_scaled = pd.DataFrame(scaler_fold.transform(X_val_fold), columns=feature_names)

    model = lgb.LGBMRegressor(**best_params)
    model.fit(X_train_fold_scaled.values, y_train_fold)

    y_pred_fold = model.predict(X_val_fold_scaled.values)
    y_pred_train_fold = model.predict(X_train_fold_scaled.values)
    
    lower_bound_fold, upper_bound_fold, _ = calculate_confidence_interval(
        model, X_val_fold_scaled.values, y_val_fold.values, y_pred_fold
    )

    val_mae = mean_absolute_error(y_val_fold, y_pred_fold)
    val_rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
    val_r2 = r2_score(y_val_fold, y_pred_fold)
    val_r = np.corrcoef(y_val_fold, y_pred_fold)[0][1]
    
    train_mae = mean_absolute_error(y_train_fold, y_pred_train_fold)
    train_rmse = np.sqrt(mean_squared_error(y_train_fold, y_pred_train_fold))
    train_r2 = r2_score(y_train_fold, y_pred_train_fold)
    train_r = np.corrcoef(y_train_fold, y_pred_train_fold)[0][1]

    fold_metrics.append({
        'Fold': fold,
        'Val_MAE': val_mae,
        'Val_RMSE': val_rmse,
        'Val_R²': val_r2,
        'Val_Pearson_r': val_r,
        'Train_MAE': train_mae,
        'Train_RMSE': train_rmse,
        'Train_R²': train_r2,
        'Train_Pearson_r': train_r,
        'CI_width_95': np.mean(upper_bound_fold - lower_bound_fold),
    })

    logging.info(f"Fold {fold} - Validation set: R²={val_r2:.4f}, MAE={val_mae:.4f}, RMSE={val_rmse:.4f}, r={val_r:.4f}")
    logging.info(f"Fold {fold} - Training set: R²={train_r2:.4f}, MAE={train_mae:.4f}, RMSE={train_rmse:.4f}, r={train_r:.4f}")

    all_y_true.extend(y_val_fold)
    all_y_pred.extend(y_pred_fold)
    all_lower_bounds.extend(lower_bound_fold)
    all_upper_bounds.extend(upper_bound_fold)
    all_indices.extend(val_idx)

    fold_results_df = pd.DataFrame({
        'No.': data_df.loc[val_idx, 'No.'].values,
        'Acceptor': data_df.loc[val_idx, 'Acceptor'].values,
        'True Value': y_val_fold,
        'Predicted Value': y_pred_fold,
        '95% CI Lower Bound': lower_bound_fold,
        '95% CI Upper Bound': upper_bound_fold,
    })
    fold_results_path = os.path.join(FOLD_RESULTS_DIR, f"fold_{fold}_results.csv")
    fold_results_df.to_csv(fold_results_path, index=False)
    logging.info(f"Fold {fold} results saved to: {fold_results_path}")

cv_results_df = pd.DataFrame({
    'No.': data_df.loc[all_indices, 'No.'].values,
    'Acceptor': data_df.loc[all_indices, 'Acceptor'].values,
    'True Value': all_y_true,
    'Predicted Value': all_y_pred,
    '95% CI Lower Bound': all_lower_bounds,
    '95% CI Upper Bound': all_upper_bounds,
})
cv_results_path = os.path.join(CROSS_VALIDATION_DIR, "cross_validation_results.csv")
cv_results_df.to_csv(cv_results_path, index=False)
logging.info(f"\nCross-validation summary results saved to: {cv_results_path}")

results_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'X_train_scaled': X_train_scaled,
    'X_test_scaled': X_test_scaled,
    'y_pred_train': y_pred_train,
    'y_pred_test': y_pred_test,
    'lower_bound_train': lower_bound_train,
    'upper_bound_train': upper_bound_train,
    'lower_bound_test': lower_bound_test,
    'upper_bound_test': upper_bound_test,
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'fold_metrics': fold_metrics,
    'all_y_true': all_y_true,
    'all_y_pred': all_y_pred,
    'all_lower_bounds': all_lower_bounds,
    'all_upper_bounds': all_upper_bounds,
    'final_model': final_model,
    'scaler': scaler,
    'best_params': best_params,
    'feature_names': feature_names,
    'data_df': data_df
}

save_results_data(results_data)

2025-10-11 16:09:46,197 - INFO - 数据集加载成功，共169条记录，6个特征
2025-10-11 16:09:46,199 - WARNING - 未找到 best_params.json，使用默认参数
2025-10-11 16:09:46,201 - INFO - 训练集大小: 152, 测试集大小: 17
2025-10-11 16:09:46,236 - INFO - 模型已保存至: saved_models\lightgbm_model.joblib
2025-10-11 16:09:46,236 - INFO - 标准化器已保存至: saved_models\scaler.joblib
2025-10-11 16:09:46,258 - INFO - 
测试集性能指标: R²=0.7545, MAE=0.0227, RMSE=0.0343, r=0.9025
2025-10-11 16:09:46,259 - INFO - 训练集性能指标: R²=0.7582, MAE=0.0192, RMSE=0.0297, r=0.8712
2025-10-11 16:09:46,262 - INFO - 
开始10折交叉验证...
2025-10-11 16:09:46,264 - INFO - 
==================== 第 1 折 ====================
2025-10-11 16:09:46,292 - INFO - 第 1 折 - 测试集: R²=0.7746, MAE=0.0231, RMSE=0.0332, r=0.8887
2025-10-11 16:09:46,293 - INFO - 第 1 折 - 训练集: R²=0.7546, MAE=0.0196, RMSE=0.0298, r=0.8689
2025-10-11 16:09:46,299 - INFO - 第 1 折结果已保存至: fold_results\fold_1_results.csv
2025-10-11 16:09:46,300 - INFO - 
==================== 第 2 折 ====================
2025-10-11 16:09:46,335 - INFO - 第 


===== 特征列和目标列信息 =====
目标列: ET1
特征列数量: 3
特征列列表: ['new-AL', 'new-AR', 'new-D']



2025-10-11 16:09:46,375 - INFO - 第 3 折 - 训练集: R²=0.8438, MAE=0.0175, RMSE=0.0250, r=0.9188
2025-10-11 16:09:46,380 - INFO - 第 3 折结果已保存至: fold_results\fold_3_results.csv
2025-10-11 16:09:46,381 - INFO - 
==================== 第 4 折 ====================
2025-10-11 16:09:46,414 - INFO - 第 4 折 - 测试集: R²=0.7011, MAE=0.0236, RMSE=0.0298, r=0.8373
2025-10-11 16:09:46,415 - INFO - 第 4 折 - 训练集: R²=0.7627, MAE=0.0190, RMSE=0.0301, r=0.8735
2025-10-11 16:09:46,420 - INFO - 第 4 折结果已保存至: fold_results\fold_4_results.csv
2025-10-11 16:09:46,421 - INFO - 
==================== 第 5 折 ====================
2025-10-11 16:09:46,455 - INFO - 第 5 折 - 测试集: R²=0.5325, MAE=0.0178, RMSE=0.0280, r=0.7388
2025-10-11 16:09:46,456 - INFO - 第 5 折 - 训练集: R²=0.8471, MAE=0.0175, RMSE=0.0245, r=0.9207
2025-10-11 16:09:46,460 - INFO - 第 5 折结果已保存至: fold_results\fold_5_results.csv
2025-10-11 16:09:46,461 - INFO - 
==================== 第 6 折 ====================
2025-10-11 16:09:46,495 - INFO - 第 6 折 - 测试集: R²=0.7564, MAE=0.02

In [4]:
import os
import json
import pandas as pd
import numpy as np

class FloatEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.float64):
            return float(obj)
        return json.JSONEncoder.default(self, obj)

try:
    results_data = load_results_data()
    
    X_train = results_data['X_train']
    X_test = results_data['X_test']
    y_train = results_data['y_train']
    y_test = results_data['y_test']
    y_pred_train = results_data['y_pred_train']
    y_pred_test = results_data['y_pred_test']
    lower_bound_train = results_data['lower_bound_train']
    upper_bound_train = results_data['upper_bound_train']
    lower_bound_test = results_data['lower_bound_test']
    upper_bound_test = results_data['upper_bound_test']
    train_metrics = results_data['train_metrics']
    test_metrics = results_data['test_metrics']
    fold_metrics = results_data['fold_metrics']
    data_df = results_data.get('data_df', None)
    
    logging.info("Successfully loaded model result data")
except NameError:
    logging.error("Model result data not found, please run Module 2 first")
    raise

if data_df is not None:
    test_results_df = pd.DataFrame({
        'No.': data_df.loc[y_test.index, 'No.'].values,
        'Acceptor': data_df.loc[y_test.index, 'Acceptor'].values,
        'True Value': y_test,
        'Predicted Value': y_pred_test,
        '95% CI Lower Bound': lower_bound_test,
        '95% CI Upper Bound': upper_bound_test
    })
else:
    test_results_df = pd.DataFrame({
        'True Value': y_test,
        'Predicted Value': y_pred_test,
        '95% CI Lower Bound': lower_bound_test,
        '95% CI Upper Bound': upper_bound_test
    })

test_results_path = os.path.join(FOLD_RESULTS_DIR, "test_results.csv")
test_results_df.to_csv(test_results_path, index=False)
logging.info(f"Test set results saved to: {test_results_path}")

if data_df is not None:
    train_results_df = pd.DataFrame({
        'No.': data_df.loc[y_train.index, 'No.'].values,
        'Acceptor': data_df.loc[y_train.index, 'Acceptor'].values,
        'True Value': y_train,
        'Predicted Value': y_pred_train,
        '95% CI Lower Bound': lower_bound_train,
        '95% CI Upper Bound': upper_bound_train
    })
else:
    train_results_df = pd.DataFrame({
        'True Value': y_train,
        'Predicted Value': y_pred_train,
        '95% CI Lower Bound': lower_bound_train,
        '95% CI Upper Bound': upper_bound_train
    })

train_results_path = os.path.join(FOLD_RESULTS_DIR, "train_results.csv")
train_results_df.to_csv(train_results_path, index=False)
logging.info(f"Training set results saved to: {train_results_path}")

cv_metrics_df = pd.DataFrame(fold_metrics)
cv_metrics_path = os.path.join(CROSS_VALIDATION_DIR, "cv_fold_metrics.csv")
cv_metrics_df.to_csv(cv_metrics_path, index=False)
logging.info(f"10-fold cross-validation detailed metrics saved to: {cv_metrics_path}")

cv_summary = {
    'Mean_Val_MAE': cv_metrics_df['Val_MAE'].mean(),
    'Std_Val_MAE': cv_metrics_df['Val_MAE'].std(),
    'Mean_Val_RMSE': cv_metrics_df['Val_RMSE'].mean(),
    'Std_Val_RMSE': cv_metrics_df['Val_RMSE'].std(),
    'Mean_Val_R²': cv_metrics_df['Val_R²'].mean(),
    'Std_Val_R²': cv_metrics_df['Val_R²'].std(),
    'Mean_Val_Pearson_r': cv_metrics_df['Val_Pearson_r'].mean(),
    'Std_Val_Pearson_r': cv_metrics_df['Val_Pearson_r'].std(),
    'Mean_Train_MAE': cv_metrics_df['Train_MAE'].mean(),
    'Std_Train_MAE': cv_metrics_df['Train_MAE'].std(),
    'Mean_Train_RMSE': cv_metrics_df['Train_RMSE'].mean(),
    'Std_Train_RMSE': cv_metrics_df['Train_RMSE'].std(),
    'Mean_Train_R²': cv_metrics_df['Train_R²'].mean(),
    'Std_Train_R²': cv_metrics_df['Train_R²'].std(),
    'Mean_Train_Pearson_r': cv_metrics_df['Train_Pearson_r'].mean(),
    'Std_Train_Pearson_r': cv_metrics_df['Train_Pearson_r'].std(),
    'Mean_CI_width_95': cv_metrics_df['CI_width_95'].mean(),
    'Std_CI_width_95': cv_metrics_df['CI_width_95'].std()
}

cv_summary_path = os.path.join(CROSS_VALIDATION_DIR, "cv_summary_metrics.csv")
pd.DataFrame([cv_summary]).to_csv(cv_summary_path, index=False)
logging.info(f"Cross-validation summary metrics saved to: {cv_summary_path}")

metrics_path = os.path.join(CROSS_VALIDATION_DIR, "performance_metrics.json")
all_metrics = {
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'fold_metrics': fold_metrics,
    'cv_summary_metrics': cv_summary
}

with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=4, cls=FloatEncoder)
logging.info(f"Performance metrics saved to: {metrics_path}")

print("\n===== 10-Fold Cross-Validation Metrics by Fold =====")
print(cv_metrics_df)

print("\n===== 10-Fold Cross-Validation Summary Metrics =====")
summary_df = pd.DataFrame([cv_summary])
print(summary_df)

2025-10-11 16:09:46,734 - INFO - 成功加载模型结果数据
2025-10-11 16:09:46,744 - INFO - 测试集结果已保存至: fold_results\test_results.csv
2025-10-11 16:09:46,753 - INFO - 训练集结果已保存至: fold_results\train_results.csv
2025-10-11 16:09:46,756 - INFO - 十折交叉验证详细指标已保存至: cross_validation_results\cv_fold_metrics.csv
2025-10-11 16:09:46,761 - INFO - 交叉验证汇总指标已保存至: cross_validation_results\cv_summary_metrics.csv
2025-10-11 16:09:46,763 - INFO - 性能指标已保存至: cross_validation_results\performance_metrics.json



===== 十折交叉验证各折性能指标 =====
   Fold   Val_MAE  Val_RMSE    Val_R²  Val_Pearson_r  Train_MAE  Train_RMSE  \
0     1  0.023064  0.033181  0.774590       0.888685   0.019620    0.029755   
1     2  0.019045  0.025629  0.528764       0.889002   0.017645    0.024760   
2     3  0.015388  0.019559  0.769668       0.880915   0.017524    0.024981   
3     4  0.023625  0.029848  0.701092       0.837314   0.019023    0.030139   
4     5  0.017801  0.027965  0.532495       0.738773   0.017497    0.024526   
5     6  0.021774  0.035716  0.756354       0.924128   0.017528    0.026314   
6     7  0.020463  0.026206  0.739125       0.889073   0.018265    0.027584   
7     8  0.020320  0.024890  0.785737       0.904894   0.018600    0.027884   
8     9  0.030098  0.042989  0.742735       0.906989   0.018507    0.029028   
9    10  0.030551  0.047759  0.426697       0.682501   0.018461    0.027618   

   Train_R²  Train_Pearson_r  CI_width_95  
0  0.754612         0.868900     0.144436  
1  0.845680     

In [5]:
# Module 4: Visualization Section

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
try:
    import shap  
except ImportError:
    shap = None
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Get result data from module 2
try:
    results_data = load_results_data()
    
    # Extract required result data
    y_train = results_data['y_train']
    y_test = results_data['y_test']
    y_pred_train = results_data['y_pred_train']
    y_pred_test = results_data['y_pred_test']
    lower_bound_train = results_data['lower_bound_train']
    upper_bound_train = results_data['upper_bound_train']
    lower_bound_test = results_data['lower_bound_test']
    upper_bound_test = results_data['upper_bound_test']
    fold_metrics = results_data['fold_metrics']
    all_y_true = results_data['all_y_true']
    all_y_pred = results_data['all_y_pred']
    all_lower_bounds = results_data['all_lower_bounds']
    all_upper_bounds = results_data['all_upper_bounds']
    X_test_scaled = results_data['X_test_scaled']
    feature_names = results_data.get('feature_names', X_test_scaled.columns.tolist())
    data_df = results_data.get('data_df', None)
    feature_importance = results_data.get('feature_importance', None)  # Get XGBoost feature importance
    
    logging.info("Successfully loaded data required for visualization")
except NameError:
    logging.error("Data required for visualization not found, please run module 2 first")
    raise

# Scatter plot function
def plot_scatter(y_true, y_pred, lower_bound, upper_bound,
                title, xlabel, ylabel, save_path, show_ci=True, color_map=COLORS['cmap']):
    """Draw scatter plot, with option to show confidence intervals"""
    fig, ax = plt.subplots(figsize=(3.3, 3))
    coef = np.polyfit(y_true, y_pred, 1)
    trendline = np.poly1d(coef)
    
    # Calculate axis range
    all_values = np.concatenate([y_true, y_pred])
    if show_ci:
        all_values = np.concatenate([all_values, lower_bound, upper_bound])
    
    data_min = np.min(all_values)
    data_max = np.max(all_values)
    data_range = data_max - data_min
    margin = data_range * 0.1
    min_val = data_min - margin
    max_val = data_max + margin
    
    # Ensure sufficient margin
    if data_range < 0.1:
        min_val = data_min - 0.05
        max_val = data_max + 0.05
    
    # Draw scatter plot
    scatter = ax.scatter(
        y_true, y_pred,
        s=60 * np.abs(y_true - y_pred) / np.max(np.abs(y_true - y_pred)) + 15,
        c=np.abs(y_true - y_pred),
        cmap=color_map,
        alpha=0.7,
        edgecolors='w',
        linewidths=0.5,
        vmin=0, vmax=np.percentile(np.abs(y_true - y_pred), 95)
    )
    
    # Draw trend line
    ax.plot(y_true, trendline(y_true), color=COLORS['secondary'], lw=1.5, linestyle='-',
            label=f'Trend line = {coef[0]:.4f}x + {coef[1]:.4f}')
    
    # Draw confidence intervals (if needed)
    if show_ci:
        residuals = y_true - y_pred
        std_error = np.std(residuals)
        x_range = np.array([min_val, max_val])
        upper_line = trendline(x_range) + 1.96 * std_error
        lower_line = trendline(x_range) - 1.96 * std_error
        ax.plot(x_range, upper_line, color=COLORS['primary'], lw=1, linestyle='--', alpha=0.7, label='95% CI')
        ax.plot(x_range, lower_line, color=COLORS['primary'], lw=1, linestyle='--', alpha=0.7)
    
    # Draw ideal line (y=x)
    ax.plot([min_val, max_val], [min_val, max_val],
            color=COLORS['neutral'], lw=1, linestyle=':', label='Ideal line (y=x)')
    
    # Set axes
    ax.set_xlabel(xlabel, labelpad=3)
    ax.set_ylabel(ylabel, labelpad=3)
    ax.xaxis.set_tick_params(width=0.5)
    ax.yaxis.set_tick_params(width=0.5)
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    
    # Add performance metrics text
    metrics_plot = {
        'R²': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r': np.corrcoef(y_true, y_pred)[0][1]
    }
    textstr = '\n'.join([
        f'$R^2$ = {metrics_plot["R²"]:.4f}',
        f'MAE = {metrics_plot["MAE"]:.4f}',
        f'RMSE = {metrics_plot["RMSE"]:.4f}',
        f'r = {metrics_plot["r"]:.4f}'
    ])
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, ha='left', va='top',
            fontsize=8, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.3'))
    
    # Add legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='lower right', frameon=True, framealpha=0.9, fontsize=6)
    
    # Add color bar
    cbar = fig.colorbar(scatter, ax=ax, pad=0.03, aspect=30)
    cbar.set_label('Absolute Error', rotation=270, labelpad=10, fontsize=8)
    cbar.ax.tick_params(labelsize=6)
    cbar.outline.set_linewidth(0.5)
    
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.title(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    logging.info(f"Chart saved to: {save_path}")

# Draw test set scatter plot with confidence bands
test_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "test_scatter_with_confidence.png")
plot_scatter(
    y_test, y_pred_test, lower_bound_test, upper_bound_test,
    "Test Set Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    test_scatter_with_ci_path,
    show_ci=True
)

# Draw test set scatter plot without confidence bands
test_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "test_scatter_without_confidence.png")
plot_scatter(
    y_test, y_pred_test, None, None,
    "Test Set Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    test_scatter_without_ci_path,
    show_ci=False
)

# Draw training set scatter plot with confidence bands
train_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "train_scatter_with_confidence.png")
plot_scatter(
    y_train, y_pred_train, lower_bound_train, upper_bound_train,
    "Training Set Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    train_scatter_with_ci_path,
    show_ci=True
)

# Draw training set scatter plot without confidence bands
train_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "train_scatter_without_confidence.png")
plot_scatter(
    y_train, y_pred_train, None, None,
    "Training Set Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    train_scatter_without_ci_path,
    show_ci=False
)

# Draw cross-validation boxplot
cv_boxplot_path = os.path.join(PLOT_SAVE_DIR, "cv_boxplot.png")
plt.rcParams.update({'font.size': 8})
fig, axes = plt.subplots(1, 3, figsize=(9.9, 3))

# MAE and RMSE boxplot
data1 = {'Validation MAE': [m['Val_MAE'] for m in fold_metrics], 
         'Validation RMSE': [m['Val_RMSE'] for m in fold_metrics]}
df1 = pd.DataFrame(data1)
sns.boxplot(data=df1, palette=[COLORS['primary'], COLORS['secondary']], 
            width=0.6, linewidth=0.7, ax=axes[0])
axes[0].set_ylabel('Value', labelpad=2)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[0].xaxis.grid(False)

# R² and Pearson correlation coefficient boxplot
data2 = {'Validation $R^2$': [m['Val_R²'] for m in fold_metrics], 
         'Validation Pearson r': [m['Val_Pearson_r'] for m in fold_metrics]}
df2 = pd.DataFrame(data2)
sns.boxplot(data=df2, palette=[COLORS['primary'], COLORS['secondary']], 
            width=0.6, linewidth=0.7, ax=axes[1])
axes[1].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[1].xaxis.grid(False)

# 95% confidence interval width boxplot
data3 = {'95% CI Width': [m['CI_width_95'] for m in fold_metrics]}
df3 = pd.DataFrame(data3)
sns.boxplot(data=df3, palette=[COLORS['primary']], 
            width=0.6, linewidth=0.7, ax=axes[2])
axes[2].set_ylabel('95% CI Width', labelpad=2)
axes[2].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[2].xaxis.grid(False)

# Add medians on boxplot
for i, col in enumerate(df1.columns):
    median = df1[col].median()
    axes[0].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

for i, col in enumerate(df2.columns):
    median = df2[col].median()
    axes[1].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

for i, col in enumerate(df3.columns):
    median = df3[col].median()
    axes[2].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

# Add title
fig.suptitle('10-Fold Cross-Validation Metrics', fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit title
plt.savefig(cv_boxplot_path, dpi=600, bbox_inches='tight')
plt.close()
logging.info(f"Cross-validation boxplot saved to: {cv_boxplot_path}")

# Draw training vs test set metrics comparison chart
comparison_plot_path = os.path.join(PLOT_SAVE_DIR, "train_vs_val_comparison.png")
plt.figure(figsize=(8, 4))

# Extract R² and MAE from training and test sets for comparison
metrics_to_plot = ['R²', 'MAE']
x = np.arange(len(metrics_to_plot))  # Label positions
width = 0.35  # Bar width

# Prepare data
val_metrics = [
    np.mean([m['Val_R²'] for m in fold_metrics]),
    np.mean([m['Val_MAE'] for m in fold_metrics])
]

train_metrics = [
    np.mean([m['Train_R²'] for m in fold_metrics]),
    np.mean([m['Train_MAE'] for m in fold_metrics])
]

# Draw bar chart
rects1 = plt.bar(x - width/2, val_metrics, width, label='Validation', color=COLORS['primary'])
rects2 = plt.bar(x + width/2, train_metrics, width, label='Training', color=COLORS['secondary'])

# Add text labels
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=7)

autolabel(rects1)
autolabel(rects2)

# Add labels and title
plt.xticks(x, metrics_to_plot)
plt.ylabel('Value')
plt.title('Training vs Validation Metrics Comparison')
plt.legend()

plt.tight_layout()
plt.savefig(comparison_plot_path, dpi=600, bbox_inches='tight')
plt.close()
logging.info(f"Training vs test set metrics comparison chart saved to: {comparison_plot_path}")

# Draw cross-validation summary scatter plot
cv_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "cv_scatter_with_confidence.png")
plot_scatter(
    np.array(all_y_true), np.array(all_y_pred), 
    np.array(all_lower_bounds), np.array(all_upper_bounds),
    "Cross-Validation Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    cv_scatter_with_ci_path,
    show_ci=True
)

cv_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "cv_scatter_without_confidence.png")
plot_scatter(
    np.array(all_y_true), np.array(all_y_pred),
    None, None,
    "Cross-Validation Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    cv_scatter_without_ci_path,
    show_ci=False
)

# Draw XGBoost feature importance bar chart (specific visualization)
if feature_importance is not None:
    importance_plot_path = os.path.join(PLOT_SAVE_DIR, "xgboost_feature_importance.png")
    plt.figure(figsize=(8, 6))
    
    # Sort by importance
    importance_sorted = feature_importance.sort_values('Importance', ascending=True)
    
    # Create horizontal bar chart
    plt.barh(importance_sorted['Feature'], importance_sorted['Importance'], 
             color=COLORS['primary'])
    
    # Add value labels (keep 4 decimal places)
    for i, v in enumerate(importance_sorted['Importance']):
        plt.text(v, i, f' {v:.4f}', va='center', fontsize=7)
    
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title('XGBoost Feature Importance')
    plt.tight_layout()
    plt.savefig(importance_plot_path, dpi=600, bbox_inches='tight')
    plt.close()
    logging.info(f"XGBoost feature importance bar chart saved to: {importance_plot_path}")

# If SHAP library is installed, draw SHAP value summary plot (XGBoost specific)
if shap is not None and 'X_test_scaled' in locals():
    try:
        # Load complete model (previously saved as serialized data, need to retrain or load complete model here)
        import joblib
        model = joblib.load(os.path.join(MODEL_SAVE_DIR, "xgboost_model.joblib"))
        
        # Calculate SHAP values
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test_scaled.values)
        
        # Draw SHAP summary plot
        shap_plot_path = os.path.join(PLOT_SAVE_DIR, "xgboost_shap_summary.png")
        plt.figure(figsize=(8, 6))
        shap.summary_plot(shap_values, X_test_scaled.values, feature_names=feature_names, 
                         plot_type="bar", show=False)
        plt.tight_layout()
        plt.savefig(shap_plot_path, dpi=600, bbox_inches='tight')
        plt.close()
        logging.info(f"XGBoost SHAP value summary plot saved to: {shap_plot_path}")
    except Exception as e:
        logging.warning(f"Failed to draw SHAP value plot: {e}")

# Correlation heatmap
def plot_correlation_heatmap(data, feature_names, save_path):
    """Draw feature correlation heatmap"""
    try:
        # Extract features and target variable
        heatmap_df = data[feature_names + ['ET1']]
        corr = heatmap_df.corr()

        fig, ax = plt.subplots(figsize=(7, 6), dpi=600)
        cmap = COLORS['cmap']
        norm = plt.Normalize(vmin=-1, vmax=1)

        # Draw correlation heatmap
        for i in range(len(corr.columns)):
            for j in range(len(corr.columns)):
                val = corr.iloc[i, j]
                if i > j:  # Lower triangle with bubbles
                    ax.scatter(
                        j, i, s=500 * abs(val),
                        c=[cmap(norm(val))], alpha=0.9,
                        edgecolors='w', linewidths=0.5
                    )
                elif i < j:  # Upper triangle with text
                    text_color = 'white' if abs(val) > 0.5 else 'black'
                    ax.text(
                        j, i, f'{val:.4f}',
                        ha='center', va='center', color=text_color, fontsize=7,
                        bbox={'facecolor': cmap(norm(val)), 'alpha': 0.8, 'pad': 2}
                    )

        # Set axes
        ax.set_xticks(np.arange(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(np.arange(len(corr.columns)))
        ax.set_yticklabels(corr.columns, fontsize=7)

        # Add grid lines
        ax.set_xticks(np.arange(len(corr.columns)) - 0.5, minor=True)
        ax.set_yticks(np.arange(len(corr.columns)) - 0.5, minor=True)
        ax.grid(which='minor', color='lightgray', linestyle='-', linewidth=0.5)

        # Add color bar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.8, aspect=30)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label('Pearson Correlation', fontsize=8)

        plt.title("Feature Correlation Matrix", fontsize=10, pad=15)
        plt.tight_layout()
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()
        logging.info(f"Correlation heatmap saved to: {save_path}")
    except Exception as e:
        logging.error(f"Failed to draw correlation heatmap: {e}")

# Draw correlation heatmap (if data available)
if data_df is not None:
    corr_heatmap_path = os.path.join(PLOT_SAVE_DIR, 'corr_heatmap.png')
    plot_correlation_heatmap(data_df, feature_names, corr_heatmap_path)

logging.info("\nAll visualization charts generated successfully")

2025-10-11 16:09:47,267 - INFO - 成功加载可视化所需数据
2025-10-11 16:09:47,942 - INFO - 带置信带图表已保存至: saved_plots\test_scatter_with_confidence.png
2025-10-11 16:09:48,501 - INFO - 无置信带图表已保存至: saved_plots\test_scatter_without_confidence.png
2025-10-11 16:09:49,038 - INFO - 带置信带图表已保存至: saved_plots\train_scatter_with_confidence.png
2025-10-11 16:09:49,568 - INFO - 无置信带图表已保存至: saved_plots\train_scatter_without_confidence.png
2025-10-11 16:09:50,502 - INFO - 交叉验证箱线图已保存至: saved_plots\cv_boxplot.png
2025-10-11 16:09:51,089 - INFO - 训练集与测试集指标对比图已保存至: saved_plots\train_vs_val_comparison.png
2025-10-11 16:09:51,637 - INFO - 带置信带图表已保存至: saved_plots\cv_scatter_with_confidence.png
2025-10-11 16:09:52,173 - INFO - 无置信带图表已保存至: saved_plots\cv_scatter_without_confidence.png
2025-10-11 16:09:52,174 - INFO - 
开始SHAP分析...
2025-10-11 16:09:52,175 - ERROR - SHAP分析失败: [Errno 2] No such file or directory: 'saved_models\\gbdt_model.joblib'
2025-10-11 16:09:52,176 - ERROR - 请确保shap库已正确安装且有足够的内存
2025-10-11 16:09:53,043 - IN

In [6]:
# Module 5: Model Invocation

import os
import json
import pandas as pd
import numpy as np
from joblib import load
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

def load_model_and_scaler(model_dir=MODEL_SAVE_DIR):
    """Load the trained model and scaler"""
    model_path = os.path.join(model_dir, "xgboost_model.joblib")
    scaler_path = os.path.join(model_dir, "scaler.joblib")
    
    try:
        model = load(model_path)
        scaler = load(scaler_path)
        logging.info(f"Successfully loaded model: {model_path}")
        logging.info(f"Successfully loaded scaler: {scaler_path}")
        return model, scaler
    except FileNotFoundError as e:
        logging.error(f"Model file not found: {e}")
        raise
    except Exception as e:
        logging.error(f"Error loading model: {e}")
        raise

def predict_new_data(model, scaler, new_data, feature_names):
    """Predict new data using the model"""
    try:
        # Ensure the feature order of new data matches the training data
        new_data_ordered = new_data[feature_names]
        
        # Standardize new data (XGBoost is not sensitive to feature scaling, but maintain consistency with training process)
        new_data_scaled = scaler.transform(new_data_ordered)
        
        # Make predictions
        predictions = model.predict(new_data_scaled)
        
        # Calculate confidence intervals
        try:
            # Get training data predictions and true values from saved results
            results_data = load_results_data()
            residuals_std = np.std(results_data['y_pred_train'] - results_data['y_train'])
        except Exception as e:
            logging.warning(f"Unable to calculate residual standard deviation, using default value: {e}")
            residuals_std = 0.5  # Provide a reasonable default value
            
        n = len(new_data)
        t_value = stats.t.ppf(0.975, n - 2)  # 95% confidence interval
        prediction_interval = t_value * residuals_std * np.sqrt(1 + 1/n)
        
        lower_bound = predictions - prediction_interval
        upper_bound = predictions + prediction_interval
        
        return predictions, lower_bound, upper_bound
    except KeyError as e:
        logging.error(f"Missing required features in new data: {e}")
        raise
    except Exception as e:
        logging.error(f"Error predicting new data: {e}")
        raise

def load_new_data(file_path=INPUT_CSV_PATH):
    """Load new data for prediction"""
    try:
        new_data = pd.read_csv(file_path, encoding='gbk')
        logging.info(f"Successfully loaded new data: {file_path}, with {len(new_data)} records")
        return new_data
    except FileNotFoundError:
        logging.error(f"New data file not found: {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading new data: {e}")
        raise

def save_predictions(new_data, predictions, lower_bound, upper_bound, output_path=OUTPUT_CSV_PATH):
    """Save prediction results"""
    try:
        # Create results DataFrame
        result_df = new_data.copy()
        result_df['Predicted ET1'] = predictions
        result_df['95% CI Lower Bound'] = lower_bound
        result_df['95% CI Upper Bound'] = upper_bound
        
        # Save results
        result_df.to_csv(output_path, index=False, encoding='gbk')
        logging.info(f"Prediction results saved to: {output_path}")
        return result_df
    except Exception as e:
        logging.error(f"Error saving prediction results: {e}")
        raise

# Main function: Load model and predict new data
def main():
    # Get feature names from configuration (or from previous modules)
    try:
        # Try to get feature names from Module 2 results
        results_data = load_results_data()
        feature_names = results_data.get('feature_names', None)
    except NameError:
        # If Module 2 hasn't been run, get feature names from data
        try:
            _, _, _, feature_names = load_data()
        except Exception as e:
            logging.error(f"Failed to get feature names: {e}")
            # You can also manually specify feature names here
            # feature_names = ['feature1', 'feature2', ...]
            return
    
    if feature_names is None:
        logging.error("Feature names not found, please ensure data is loaded correctly")
        return
    
    # Load model and scaler
    model, scaler = load_model_and_scaler()
    
    # Load new data
    new_data = load_new_data()
    
    # Make predictions
    predictions, lower_bound, upper_bound = predict_new_data(model, scaler, new_data, feature_names)
    
    # Save prediction results
    result_df = save_predictions(new_data, predictions, lower_bound, upper_bound)
    
    # Display first 5 prediction results
    logging.info("\nFirst 5 prediction results:")
    print(result_df.head())

if __name__ == "__main__":
    main()

2025-10-11 16:09:53,073 - ERROR - 模型文件未找到: [Errno 2] No such file or directory: 'saved_models\\gbdt_model.joblib'


FileNotFoundError: [Errno 2] No such file or directory: 'saved_models\\gbdt_model.joblib'